In [1]:
using CovidSim_ilm

In [5]:
using StatsBase
using TypedTables
using BenchmarkTools
using Distributions
using YAML
using PrettyPrint
using Plots

┌ Info: Precompiling PrettyPrint [8162dcfd-2161-5ef2-ae6c-7681170c5f98]
└ @ Base loading.jl:1342


In [3]:
cd(joinpath(homedir(),"Dropbox/Covid Modeling/Covid-ILM/source"))

## Load data and params

In [6]:
# set locale
locale = 38015
ndays = 180

180

In [9]:
alldat = setup(ndays, [locale]; paramdir="../parameters", 
    geofilename="../data/geo2data.csv",
    socialfilename = "socialparams.yml",
    vaccinefilename = "vaccines.yml",
    variantsfilename = "variants.yml"
    );

In [13]:
keys(alldat)

(:dat, :transitionset, :geo, :infectset, :social, :trvec)

In [16]:
alldat.infectset[:base].recvrisk

5-element Vector{Float64}:
 0.1
 0.39
 0.44
 0.54
 0.56

In [29]:
shifter(alldat.infectset[:base].recvrisk, mult=1.2)

5-element Vector{Float64}:
 0.12
 0.468
 0.528
 0.648
 0.672

In [17]:
alldat.infectset[:base].sendrisk

25-element Vector{Float64}:
 0.0
 0.3
 0.65
 0.75
 0.85
 0.85
 0.75
 0.7
 0.65
 0.6
 0.5
 0.2
 0.1
 0.1
 0.1
 0.05
 0.05
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0

In [6]:
spreadparams = alldat["sp"]  # the spread parameters are loaded as a dict of float arrays

(send_risk = [0.0, 0.3, 0.65, 0.75, 0.85, 0.85, 0.75, 0.7, 0.65, 0.6  …  0.05, 0.05, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], recv_risk = [0.1, 0.39, 0.44, 0.54, 0.56], contact_factors = Dict{agegrp, Dict{condition, Float64}}(age40_59 => Dict(nil => 2.1, severe => 0.6, sick => 1.0, mild => 2.0), age20_39 => Dict(nil => 2.1, severe => 0.6, sick => 1.0, mild => 2.0), age60_79 => Dict(nil => 1.7, severe => 0.5, sick => 0.7, mild => 1.6), age0_19 => Dict(nil => 1.1, severe => 0.5, sick => 0.7, mild => 1.1), age80_up => Dict(nil => 1.0, severe => 0.5, sick => 0.6, mild => 0.9)), touch_factors = Dict{agegrp, Dict{Enum{Int32}, Float64}}(age40_59 => Dict(nil => 0.61, severe => 0.18, recovered => 0.61, sick => 0.3, unexposed => 0.61, mild => 0.58), age20_39 => Dict(nil => 0.63, severe => 0.18, recovered => 0.63, sick => 0.35, unexposed => 0.63, mild => 0.62), age60_79 => Dict(nil => 0.41, severe => 0.18, recovered => 0.41, sick => 0.18, unexposed => 0.41, mild => 0.41), age0_19 => Dict(nil => 0

In [7]:
keys(spreadparams)

(:send_risk, :recv_risk, :contact_factors, :touch_factors, :shape)

In [8]:
infectfilename="../parameters/spread_params.yml"

"../parameters/spread_params.yml"

In [9]:
spread_inputs = YAML.load_file(infectfilename)

Dict{Any, Any} with 5 entries:
  "send_risk"       => [0.0, 0.3, 0.65, 0.75, 0.85, 0.85, 0.75, 0.7, 0.65, 0.6 …
  "shape"           => 1.0
  "touch_factors"   => Dict{Any, Any}(5=>Dict{Any, Any}("sick"=>0.18, "unexpose…
  "contact_factors" => Dict{Any, Any}(5=>Dict{Any, Any}("sick"=>0.6, "nil"=>1.0…
  "recv_risk"       => [0.1, 0.39, 0.44, 0.54, 0.56]

In [10]:
spr_day = 6; recv_age = 4

4

In [14]:
recvrisk = spread_inputs["recv_risk"]

5-element Vector{Float64}:
 0.1
 0.39
 0.44
 0.54
 0.56

In [15]:
sendrisk = spread_inputs["send_risk"]

25-element Vector{Float64}:
 0.0
 0.3
 0.65
 0.75
 0.85
 0.85
 0.75
 0.7
 0.65
 0.6
 0.5
 0.2
 0.1
 0.1
 0.1
 0.05
 0.05
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0

In [37]:
@btime $sendrisk[$spr_day] * $recvrisk[$recv_age]

  1.883 ns (0 allocations: 0 bytes)


0.459

In [17]:
Statcond = Union{status, condition}

Union{condition, status}

In [18]:
foo = Array{Statcond, 1}

Vector{Union{condition, status}} (alias for Array{Union{condition, status}, 1})

In [19]:
foo = [nil, dead]

2-element Vector{Enum{Int32}}:
 nil::condition = 5
 dead::status = 4